In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
from pathlib import Path
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode
import numpy as np

# Scraping

In [ ]:
# URL da vítima
url = "https://quotes.toscrape.com/page/1/"

# Cabeçalho para simular um navegador
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [ ]:
response = requests.get(url, headers=headers)
response
# <Response [200]> -> requisição foi bem-sucedida

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
# Exibe o html como texto
print(soup.get_text())

As Notícias Seguem o Seguinte Padrão:

```html
  <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”</span>
        <span>by <small class="author" itemprop="author">Albert Einstein</small>
        <a href="/author/Albert-Einstein">(about)</a>
        </span>
        <div class="tags">
            Tags:
            <meta class="keywords" itemprop="keywords" content="inspirational,life,live,miracle,miracles" /    >
            
            <a class="tag" href="/tag/inspirational/page/1/">inspirational</a>
            
            <a class="tag" href="/tag/life/page/1/">life</a>
            
            <a class="tag" href="/tag/live/page/1/">live</a>
            
            <a class="tag" href="/tag/miracle/page/1/">miracle</a>
            
            <a class="tag" href="/tag/miracles/page/1/">miracles</a>
            
        </div>
    </div>


In [ ]:
# Localizando o bloco com o id "textoMateria"
bloco_noticias = soup.findAll("div", class_="quote")

bloco_noticias

In [ ]:
# Extraindo a frase
frases = soup.find_all (class_="text")
for a in frases:
    print(a.text)

In [ ]:
# Extraindo o autor
autores = soup.find_all("small", class_="author")
for a in autores:
    print(a.text)

In [ ]:
bloco_tags = soup.findAll("div", class_="tags")

for bloco in bloco_tags:
    tags = bloco.find_all("a", class_="tag")
    lista_tags = [tag.text for tag in tags]
    print(lista_tags)

In [ ]:
numero_paginas = 40

dados = []

for p in tqdm(range(1, numero_paginas)):
    url = f"https://quotes.toscrape.com/page/{p}/"
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    quotes = soup.find_all("div", class_="quote")

    for q in quotes:
        frase = q.find("span", class_="text").text
        autor = q.find("small", class_="author").text
        tags = [tag.text for tag in q.find_all("a", class_="tag")]

        dados.append({
            "frase": frase,
            "autor": autor,
            "tags": tags
        })

    time.sleep(random.uniform(0, 0.1))

In [ ]:
df = pd.DataFrame(dados)
df

In [ ]:
df.to_csv("df_quotes.csv", index=False)

# PLN

In [ ]:
df = pd.read_csv("C:/Users/isabe/Perspectiva_dados/PLN/df_quotes.csv")
print(len(df))
print(df)

### Limpeza básica

In [ ]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["frase_limpa"] = df["frase"].apply(limpar_texto)

df[["frase", "frase_limpa"]].head()

### remoção de stopwords

In [ ]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("english")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["frase_limpa"].apply(remover_stopwords)
df["frase_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["frase_limpa", "tokens_sem_stopwords", "frase_sem_stopwords"]].head()

df

# Bag of Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["frase_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

In [ ]:
# retirando os numeros
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

In [ ]:
# Criando o df final
metadados = df[["frase", "autor", "tags"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.to_csv("df_final.csv", index=False)

df_final.head(10)

# TF - IDF

In [ ]:
# Total de palavras
TP = []
TP = df_bow.sum(axis=1)
TP

In [ ]:
TF = df_bow.div(TP, axis=0)
TF

In [ ]:
df_ = (df_bow > 0).sum(axis=0)
df_

In [ ]:
N = df_bow.shape[0]
N = len(df_bow)
IDF = np.log(N / df_)
df_idf = pd.DataFrame({
    "documentos_com_o_termo": df_,
    "idf": IDF
}).sort_values("idf", ascending=False)

df_idf.head(10)

In [ ]:
frase = 0

print(df_final.loc[frase, "frase"])

TF.loc[frase].sort_values(ascending=False).head(10)

In [ ]:
df_idf.tail(10)

In [ ]:
TF_IDF = TF.mul(IDF, axis=1)

TF_IDF.head()

In [ ]:
TF_IDF.loc[0].sort_values(ascending=False).head(10)

In [ ]:
TF_IDF_pref = TF_IDF.add_prefix("tfidf_").reset_index(drop=True)

DF_TFIDF = pd.concat([metadados, TF_IDF_pref], axis=1)

DF_TFIDF.head()

In [ ]:
from pathlib import Path
# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[["frase", "autor", "tags", "frase_limpa", "frase_sem_stopwords"]]
df_completo.to_csv("Frases_completo.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv("df_bow.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
DF_TFIDF.to_csv("TF_IDF.csv", index=False)